<a href="https://colab.research.google.com/github/krzysztofnowakuz/colab/blob/main/Dog_Lifespan_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Projekt: Dog Lifespan Regression

**Cel:**  
Przewidzieć średnią długość życia rasy psa na podstawie cech fizycznych (waga, wysokość itp.).  

**Dane:**  
- Kaggle: Dog Breeds Details Dataset  
- Informacje o rasach psów: min/max wysokość, min/max waga, długość życia, rasa.  

**Metody:**  
- Eksploracyjna analiza danych (EDA)  
- Feature engineering (średnia wysokość/wagi, wskaźnik „BMI”)  
- Regresja liniowa  
- Ridge Regression  
- Lasso Regression  
- Ewaluacja: MAE, RMSE, R²  

**Plan działania:**  
1. Wczytać dane i sprawdzić ich jakość  
2. Przygotować cechy (height_mean, weight_mean, bmi_like)  
3. Podzielić dane na zbiór treningowy i testowy  
4. Zbudować modele (Linear, Ridge, Lasso)  
5. Ocenić wyniki na zbiorze testowym  
6. Zinterpretować współczynniki i podsumować wnioski


## 📦 Dane (Google Drive → gdown)

Ten notebook pobiera dane z publicznego folderu Google Drive przy pomocy **gdown**.  
Nie trzeba montować całego Drive ani używać żadnych kluczy API.

- Źródło: publiczny folder Google Drive (read-only)
- Pobieranie: `gdown` (automatyczne ściąganie wszystkich plików z folderu)
- Lokalizacja danych po pobraniu: `/content/data`

In [8]:
# --- Konfiguracja ---
DATA_FOLDER_URL = "https://drive.google.com/drive/folders/1Al4r365Ze_Oxlfr02j-b0yuYoaj4H4i7?usp=sharing"

CSV_NAME = "dog_breeds.csv"

!pip install -q gdown>=5.0.0

import os
import pandas as pd
import gdown
from pathlib import Path

OUT_DIR = Path("/content/data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not any(OUT_DIR.iterdir()):
    print("⬇️ Pobieranie danych z Google Drive (folder publiczny)...")
    gdown.download_folder(
        url=DATA_FOLDER_URL,
        output=str(OUT_DIR),
        quiet=False,
        use_cookies=False
    )
else:
    print("✅ Dane już pobrane.")

csv_path = OUT_DIR / CSV_NAME
if not csv_path.exists():
    raise FileNotFoundError(
        f"Nie znaleziono pliku {CSV_NAME} w folderze Google Drive.\n"
        f"Sprawdź nazwę albo zawartość folderu: {[p.name for p in OUT_DIR.iterdir()]}"
    )

print("📄 Używany plik CSV:", csv_path)
df = pd.read_csv(csv_path)
print("🔎 Kształt danych:", df.shape)
df.head()


✅ Dane już pobrane.
📄 Używany plik CSV: /content/data/dog_breeds.csv
🔎 Kształt danych: (97, 23)


,Name,min_life_expectancy,max_life_expectancy,max_height_male,max_height_female,max_weight_male,max_weight_female,min_height_male,min_height_female,min_weight_male,...,shedding,grooming,drooling,coat_length,good_with_strangers,playfulness,protectiveness,trainability,energy,barking
0,Golden Retriever,10,12,24.0,24.0,75.0,65.0,23.0,23.0,65.0,...,4,2,2,1,5,4,3,5,3,1
1,Dachshund,12,16,9.0,9.0,32.0,32.0,8.0,8.0,16.0,...,2,2,2,2,4,4,4,4,3,5
2,Labrador Retriever,10,12,24.5,24.5,80.0,70.0,22.5,22.5,65.0,...,4,2,2,1,5,5,3,5,5,3
3,Great Dane,7,10,32.0,32.0,175.0,140.0,30.0,30.0,140.0,...,3,1,4,1,3,4,5,3,4,3
4,Boxer,10,12,25.0,25.0,80.0,65.0,23.0,23.0,65.0,...,2,2,3,1,4,4,4,4,4,3


In [10]:
# Podstawowe informacje o kolumnach
data_report = pd.DataFrame({
    "Typ danych": df.dtypes,
    "Liczba braków": df.isnull().sum(),
    "Procent braków": (df.isnull().sum() / len(df) * 100).round(2),
    "Unikalne wartości": df.nunique()
})

# Posortowane po brakach
data_report = data_report.sort_values(by="Liczba braków", ascending=False)

print(f"🔎 Dataset ma {df.shape[0]} wierszy i {df.shape[1]} kolumn.")
data_report


🔎 Dataset ma 97 wierszy i 23 kolumn.


,Typ danych,Liczba braków,Procent braków,Unikalne wartości
Name,object,0,0.0,97
min_life_expectancy,int64,0,0.0,9
max_life_expectancy,int64,0,0.0,10
max_height_male,float64,0,0.0,36
max_height_female,float64,0,0.0,37
max_weight_male,float64,0,0.0,46
max_weight_female,float64,0,0.0,43
min_height_male,float64,0,0.0,35
min_height_female,float64,0,0.0,35
min_weight_male,float64,0,0.0,44


In [11]:
stats = df.describe(include="all").T
stats = stats[["count","mean","std","min","25%","50%","75%","max"]]
stats.round(2)


,count,mean,std,min,25%,50%,75%,max
Name,97,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min_life_expectancy,97.0,11.360825,1.653189,7.0,10.0,12.0,12.0,15.0
max_life_expectancy,97.0,14.0,1.870829,9.0,13.0,14.0,15.0,19.0
max_height_male,97.0,20.561856,6.949692,7.0,15.0,22.0,27.0,33.0
max_height_female,97.0,20.427835,6.836251,7.0,15.0,22.0,27.0,32.0
max_weight_male,97.0,60.890722,43.281424,6.0,23.0,60.0,85.0,200.0
max_weight_female,97.0,54.292784,36.473518,6.0,22.0,55.0,71.0,200.0
min_height_male,97.0,18.164948,6.747963,5.0,12.0,19.0,24.0,30.0
min_height_female,97.0,18.061856,6.656473,5.0,12.0,19.0,24.0,30.0
min_weight_male,97.0,46.024742,34.927431,3.0,15.0,40.0,65.0,150.0
